# ml_scan milestone gates

One markdown + one code cell per milestone. Restart-safe. Same checks as `user_command.md`.

**Date window:** this warehouse has hourly bars from **2025-09-01** onward, not 2024. Run the first code cell so `SMOKE_START` / `SMOKE_END` match live coverage, then continue from M07 if you already passed M01–M04.

**Training universe:** edit `TRAINING_UNIVERSE` in the first code cell (CSV with a `symbol` column). Feature build (M09+), diagnostics, and scan use that path. Re-run from M09 after you update the file.

**Skipped:** M05 (Nifty 500 snapshot), M06 (ADTV liquid/smoke filter), and M25 (liquid e2e) — symbol selection lives in `scan_trade` via `model_training_symbol.csv`.


In [1]:
from pathlib import Path
import pandas as pd
from ml_scan.config import load_settings
from ml_scan.data.universe import load_symbol_list
from ml_scan.storage.db import Database

ROOT = Path('.').resolve()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.expand_frame_repr', False)

# --- edit this when the training symbol list changes ---
TRAINING_UNIVERSE = Path(r"C:\Users\mail2\OneDrive\projects2\scan_trade\data\universe\model_training_symbol.csv")
# Fallback for a quick smoke run: Path("data/universe/smoke_symbols.csv")
assert TRAINING_UNIVERSE.is_file(), f"missing training universe: {TRAINING_UNIVERSE}"
TRAINING_SYMBOLS = load_symbol_list(TRAINING_UNIVERSE)

# Warehouse has no 2024 bars; derive the smoke window from live coverage.
db = Database.from_settings(load_settings())
with db.connection() as c:
    cov = c.execute('select min(ts) as mn, max(ts) as mx from ohlcv_60m').fetchone()
SMOKE_START = pd.Timestamp(cov['mn']).tz_convert('Asia/Kolkata').strftime('%Y-%m-%d')
SMOKE_END = pd.Timestamp(cov['mx']).tz_convert('Asia/Kolkata').strftime('%Y-%m-%d')
print(ROOT)
print('warehouse', SMOKE_START, '→', SMOKE_END)
print('training universe', TRAINING_UNIVERSE, f'({len(TRAINING_SYMBOLS)} symbols)')

C:\Users\mail2\OneDrive\projects2\ml_scan
warehouse 2021-09-06 → 2026-09-04
training universe C:\Users\mail2\OneDrive\projects2\scan_trade\data\universe\model_training_symbol.csv (401 symbols)


## M01 — Repo skeleton

Settings must load and Timescale DB name must be `scan_trade`.

In [2]:
import ml_scan
import ml_scan.config
print(ml_scan.config.load_settings().timescale.db)

scan_trade


## M02 — Storage copy

`ohlcv_60m` must already have rows in the existing warehouse.

In [3]:
from ml_scan.storage.db import Database
from ml_scan.config import load_settings
db = Database.from_settings(load_settings())
with db.connection() as c:
    n = c.execute('select count(*) as n from ohlcv_60m').fetchone()['n']
print('hourly_rows', n)
assert n > 0

hourly_rows 3909572


## M03 — Calendar primitives

NSE hourly index is 7 bars per session.

In [4]:
import pandas as pd
from ml_scan.data.calendar import NSECalendar
c = NSECalendar()
print(len(c.hourly_index(pd.Timestamp('2024-01-02', tz='Asia/Kolkata'))))

7


## M04 — TimescaleAdapter

Hourly read for RELIANCE; no 15:15 partial hours.

In [5]:
from ml_scan.config import load_settings
from ml_scan.data.timescale_adapter import TimescaleAdapter
a = TimescaleAdapter(load_settings())
df = a.read(['RELIANCE'], '60minute', SMOKE_START, SMOKE_END)
assert len(df) > 0, f'no hourly bars for RELIANCE between {SMOKE_START} and {SMOKE_END}'
assert {'ts','symbol','open','high','low','close','volume','interval'} <= set(df.columns)
assert (df['interval'] == '60minute').all()
ts = pd.to_datetime(df['ts'], utc=True).dt.tz_convert('Asia/Kolkata')
assert not ts.dt.strftime('%H:%M').eq('15:15').any()
print(df.head())
print('-'*70)
print(df.tail())

                         ts    symbol  instrument_token     open     high      low    close      volume source  n_5m  interval  is_partial_hour
0 2021-09-06 09:15:00+05:30  RELIANCE            738561  1150.00  1182.00  1150.00  1178.60  16041364.0   cagg    12  60minute            False
1 2021-09-06 10:15:00+05:30  RELIANCE            738561  1178.60  1178.75  1166.50  1168.55   3185310.0   cagg    12  60minute            False
2 2021-09-06 11:15:00+05:30  RELIANCE            738561  1168.50  1169.80  1161.95  1162.90   2336124.0   cagg    12  60minute            False
3 2021-09-06 12:15:00+05:30  RELIANCE            738561  1162.90  1167.20  1160.50  1165.30   2204576.0   cagg    12  60minute            False
4 2021-09-06 13:15:00+05:30  RELIANCE            738561  1165.05  1166.10  1155.00  1156.80   2857552.0   cagg    12  60minute            False
----------------------------------------------------------------------
                            ts    symbol  instrument_token    ope

## M07 — MTFAligner

Print hourly `ts` vs joined `d_close` date. Same-session daily close must not leak.

In [6]:
import subprocess, sys
import pandas as pd

start, end = SMOKE_START, SMOKE_END
subprocess.check_call([
    sys.executable, "-m", "ml_scan.cli", "data", "align",
    "--symbols", "RELIANCE,TCS",
    "--start", start, "--end", end,
    "--out", "data/artifacts/align_smoke.parquet",
])
subprocess.check_call([sys.executable, "-m", "pytest", "tests/test_mtf_aligner.py", "-q"])
al = pd.read_parquet('data/artifacts/align_smoke.parquet')
print(al[['symbol','ts','d_ts','d_close']].head(8))
print(al.shape)

     symbol                        ts                      d_ts  d_close
0  RELIANCE 2021-09-06 09:15:00+05:30                       NaT      NaN
1  RELIANCE 2021-09-06 10:15:00+05:30                       NaT      NaN
2  RELIANCE 2021-09-06 11:15:00+05:30                       NaT      NaN
3  RELIANCE 2021-09-06 12:15:00+05:30                       NaT      NaN
4  RELIANCE 2021-09-06 13:15:00+05:30                       NaT      NaN
5  RELIANCE 2021-09-06 14:15:00+05:30                       NaT      NaN
6  RELIANCE 2021-09-07 09:15:00+05:30 2021-09-06 00:00:00+05:30  1156.05
7  RELIANCE 2021-09-07 10:15:00+05:30 2021-09-06 00:00:00+05:30  1156.05
(14822, 24)


## M08 — TA engine parity

In [7]:
!python -m pytest tests/test_ta_engine_parity.py -q

.                                                                        [100%]
============================== warnings summary ===============================
tests/test_ta_engine_parity.py::test_ta_engine_matches_legacy_helper
tests/test_ta_engine_parity.py::test_ta_engine_matches_legacy_helper
  c:\Users\mail2\OneDrive\projects2\ml_scan\.ml_env\Lib\site-packages\pandas\core\window\rolling.py:611: RuntimeWarning: All-NaN slice encountered
    return func(x, start, end, min_periods, *numba_args)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 2 warnings in 3.08s


## M09 — PanelFeatureEngineer

In [8]:
import subprocess, sys
import pandas as pd

start, end = SMOKE_START, SMOKE_END
subprocess.check_call([
    sys.executable, "-m", "ml_scan.cli", "features", "hourly",
    "--universe", str(TRAINING_UNIVERSE),
    "--start", start, "--end", end,
    "--out", "data/artifacts/feat_hourly_smoke.parquet",
])
feat = pd.read_parquet('data/artifacts/feat_hourly_smoke.parquet')
assert len(feat) > 0, f'hourly features empty for {start} → {end}'
assert 'ATR' in feat.columns, 'TA columns missing'
n_syms = feat['symbol'].nunique()
assert n_syms > 0, 'no symbols in hourly panel'
meta = {'symbol', 'ts', 'interval', 'source', 'instrument_token', 'open', 'high', 'low', 'close', 'volume'}
n_ta = len([c for c in feat.columns if c not in meta])
print(f'hourly panel: {len(feat)} rows x {feat.shape[1]} columns ({n_ta} pandas_ta / derived, mode from configs/default.yaml)')
print(f'universe: {TRAINING_UNIVERSE} → {n_syms} symbols in panel')
print(feat.columns[:20].tolist())
print(feat[['symbol','ts']].head())

hourly panel: 2971299 rows x 239 columns (229 pandas_ta / derived, mode from configs/default.yaml)
universe: C:\Users\mail2\OneDrive\projects2\scan_trade\data\universe\model_training_symbol.csv → 401 symbols in panel
['ts', 'open', 'high', 'low', 'close', 'volume', 'EBSW', 'REFLEX', 'AO', 'APO', 'BIAS', 'CCI', 'CFO', 'CG', 'CMO', 'COPPOCK', 'CRSI', 'CTI', 'ER', 'BULLP_13']
   symbol                        ts
0  360ONE 2021-09-06 09:15:00+05:30
1  360ONE 2021-09-06 10:15:00+05:30
2  360ONE 2021-09-06 11:15:00+05:30
3  360ONE 2021-09-06 12:15:00+05:30
4  360ONE 2021-09-06 13:15:00+05:30


### M09b — Which stocks feed the model, and how many pandas_ta features it adds (Q1, Q2)

**Q1**: the split used later (M15b) is time-based, not stock-based, so every symbol in `TRAINING_UNIVERSE` is used identically for both training and testing — there is no separate "held-out stock" list. The table below is the per-symbol row count feeding the pipeline from this point on.

**Q2**: M09 now runs pandas_ta in **full** mode (`features.hourly_mode: full` in `configs/default.yaml`) — every compatible indicator (`generate_all_ta_features`, 200+ columns). The cell below also prints the lite-mode count so the size of that jump is visible. These are *hourly* TA columns only; M10 below adds daily and 15-minute context on top, then M12 QC drops low-quality columns (high missingness / zero variance, treating inf as missing).

In [9]:
import pandas as pd

from ml_scan.config import load_settings
from ml_scan.data.universe import load_symbol_list
from ml_scan.features.qc import symbol_row_counts
from ml_scan.features.ta_engine import generate_all_ta_features, generate_lite_ta_features

symbols = load_symbol_list(TRAINING_UNIVERSE)
preview = symbols if len(symbols) <= 20 else symbols[:20] + [f"... (+{len(symbols) - 20} more)"]
print(f"Q1 -- {len(symbols)} training-universe stocks from {TRAINING_UNIVERSE}: {preview}")

hourly = pd.read_parquet('data/artifacts/feat_hourly_smoke.parquet')
print("\nrows contributed per stock at the hourly-feature stage:")
display(symbol_row_counts(hourly)[['symbol', 'n_rows']])

meta = {'symbol', 'ts', 'interval', 'source', 'instrument_token', 'open', 'high', 'low', 'close', 'volume'}
n_in_panel = len([c for c in hourly.columns if c not in meta])
mode = load_settings().features.hourly_mode
print(f"\nQ2 -- configured hourly_mode={mode!r}: {n_in_panel} non-OHLCV columns actually written by M09")

sample = (
    hourly.loc[hourly['symbol'] == symbols[0], ['ts', 'open', 'high', 'low', 'close', 'volume']]
    .sort_values('ts')
    .set_index('ts')
)
ohlcv_cols = {'open', 'high', 'low', 'close', 'volume'}
n_lite = len(set(generate_lite_ta_features(sample).columns) - ohlcv_cols)
n_full = len(set(generate_all_ta_features(sample).columns) - ohlcv_cols)
print(f"Q2 -- pandas_ta 'lite' mode: {n_lite} new indicator columns")
print(f"Q2 -- pandas_ta 'full' mode (every compatible indicator): {n_full} new indicator columns")

Q1 -- 401 training-universe stocks from C:\Users\mail2\OneDrive\projects2\scan_trade\data\universe\model_training_symbol.csv: ['360ONE', '3MINDIA', 'AARTIIND', 'AAVAS', 'ABB', 'ABBOTINDIA', 'ABCAPITAL', 'ABFRL', 'ABREL', 'ACC', 'ACE', 'ACUTAAS', 'ADANIENSOL', 'ADANIENT', 'ADANIGREEN', 'ADANIPORTS', 'ADANIPOWER', 'AEGISLOG', 'AFFLE', 'AIAENG', '... (+381 more)']

rows contributed per stock at the hourly-feature stage:


,symbol,n_rows
0,360ONE,7411
1,3MINDIA,7411
2,AARTIIND,7411
3,AAVAS,7411
4,ABB,7411
...,...,...
396,ZENSARTECH,7411
397,ZENTEC,7411
398,ZFCVINDIA,7411
399,ZYDUSLIFE,7411



Q2 -- configured hourly_mode='full': 229 non-OHLCV columns actually written by M09
Q2 -- pandas_ta 'lite' mode: 22 new indicator columns
Q2 -- pandas_ta 'full' mode (every compatible indicator): 227 new indicator columns


## M10 — Daily + 15m join

In [10]:
!python -m ml_scan.cli features mtf --in data/artifacts/feat_hourly_smoke.parquet --out data/artifacts/feat_mtf_smoke.parquet
import pandas as pd
mtf = pd.read_parquet('data/artifacts/feat_mtf_smoke.parquet')
d_cols = [c for c in mtf.columns if c.startswith('d_')]
m15_cols = [c for c in mtf.columns if c.startswith('m15_')]
assert d_cols, 'daily columns missing'
assert m15_cols, '15-minute columns missing'
print(d_cols[:8])
print(m15_cols[:8])

wrote C:\Users\mail2\OneDrive\projects2\ml_scan\data\artifacts\feat_mtf_smoke.parquet
['d_ts', 'd_open', 'd_high', 'd_low', 'd_close', 'd_volume', 'd_ema50', 'd_ema200']
['m15_ts', 'm15_open', 'm15_high', 'm15_low', 'm15_close', 'm15_volume', 'm15_rsi', 'm15_vwap_dist']


## M11 — SwingLabeler

Print class balance. The next cell (M11b) turns that imbalance into inverse-frequency class weights.

In [ ]:
!python -m ml_scan.cli features label --in data/artifacts/feat_mtf_smoke.parquet --out data/artifacts/labeled_smoke.parquet
import pandas as pd
lab = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
assert len(lab) > 0, 'labeled panel is empty — re-run M09/M10 with warehouse dates'
print(lab['y'].value_counts(dropna=False))
print(lab['y_reason'].value_counts(dropna=False))

### M11b — Class imbalance and class weights

The swing label is typically imbalanced (more stop-hits than take-profit hits). Training without weights makes trees predict the majority class and inflates raw accuracy while crushing recall — which is why the earlier smoke metrics looked worse than they needed to.

Weights follow the reference `cwts` formula: `w_i = n / (2 * n_i)`. With those weights, both classes contribute equally (`w0 * n0 == w1 * n1 == n/2`). From here on, RandomForest / LightGBM / XGBoost, Boruta, and hyperparameter search all use these weights (computed on the *train* fold only, never the test fold).

In [ ]:
import pandas as pd

from ml_scan.ml_engine.metrics import class_weight_dict

lab = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
resolved = lab.loc[lab['y'].isin([0, 1]), 'y'].astype(int)
counts = resolved.value_counts().sort_index()
print('resolved label counts:')
print(counts.to_string())
print(f"\npositive rate (wins): {resolved.mean():.4f}")

cw = class_weight_dict(resolved)
print('\nclass weights (cwts):')
print({k: round(v, 4) for k, v in cw.items()})
print(f"weighted class 0 contribution: {cw[0] * int(counts.get(0, 0)):.4f}")
print(f"weighted class 1 contribution: {cw[1] * int(counts.get(1, 0)):.4f}")
print('(those two should be equal — both classes now pull with the same total weight)')

## M12 — Feature QC + leakage

In [ ]:
!python -m ml_scan.cli features qc --in data/artifacts/labeled_smoke.parquet --out data/artifacts/feature_qc_report.json
!python -m pytest tests/test_leakage.py -q

### M12b — Baseline model comparison on the initial feature set (Q0, Q3)

Before Boruta ever runs, fit **RandomForest, XGBoost, and LightGBM** on every QC-passed feature from the **full** pandas_ta matrix, scored on the same purged walk-forward folds used everywhere below. Each model is trained with the M11b class weights (per-row `sample_weight` from `cwts`, recomputed on that fold's training labels only).

This is the first place `accuracy`, `balanced_accuracy`, `precision`, `recall`, and `roc_auc` are all reported together (Q0), for all three candidate models (Q3). The winner (by fold-weighted ROC-AUC) is written to `data/artifacts/model_comparison_initial.json` as `best_model`, and is the model the rest of this notebook's diagnostic cells (M13b, M14b, M16a, M16b) carry forward.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from ml_scan.config import load_settings
from ml_scan.features.qc import run_qc, select_xy
from ml_scan.ml_engine.estimator import compare_models, select_best_model
from ml_scan.ml_engine.metrics import fold_metrics_table
from ml_scan.ml_engine.splitter import PurgedWalkForward

settings = load_settings()
labeled = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
_, qc_report = run_qc(labeled, missing_threshold=settings.features.missing_threshold)
initial_features = qc_report['features']
print(f"Q2/Q3 setup: {len(initial_features)} QC-passed features feed the baseline comparison")
print(initial_features)

X, y, panel = select_xy(labeled, initial_features)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
baseline_summary, baseline_models = compare_models(X, y, panel, splitter=splitter, random_state=settings.ml.random_state)
best_model = select_best_model(baseline_summary, metric='roc_auc')

print("\nQ0 / Q3 -- baseline modeling results, initial feature set, every model:")
display(baseline_summary.round(4))
print(f"\nbest model by fold-weighted ROC-AUC: {best_model!r}")
print("\nfold-by-fold detail for the winner:")
display(fold_metrics_table(baseline_models[best_model].fold_metrics_).round(4))

dest = Path('data/artifacts/model_comparison_initial.json')
dest.write_text(
    json.dumps(
        {
            'best_model': best_model,
            'n_initial_features': len(initial_features),
            'initial_features': initial_features,
            'summary': baseline_summary.reset_index().to_dict(orient='records'),
        },
        indent=2,
    ),
    encoding='utf-8',
)
print(f"\nwrote {dest}")

## M13 — Boruta

In [ ]:
!python -m ml_scan.cli ml boruta --in data/artifacts/labeled_smoke.parquet --out data/artifacts/boruta_features.json --max-iter 50

### M13b — Boruta on the best model + modeling results (Q4)

Re-runs Boruta with `estimator_name=best_model` (the M12b winner) as the shadow-feature classifier — the M13 gate cell above always uses the configured default (`ml.model` in `configs/default.yaml`), which happens to be the same model here but will not always be. Reports how many of the initial features survive, and re-fits `best_model` on just those features so the effect of feature selection is visible before VIF runs.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from ml_scan.config import load_settings
from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.estimator import compare_models
from ml_scan.ml_engine.metrics import fold_metrics_table
from ml_scan.ml_engine.selector import BorutaSelector
from ml_scan.ml_engine.splitter import PurgedWalkForward

settings = load_settings()
labeled = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
init = json.loads(Path('data/artifacts/model_comparison_initial.json').read_text())
best_model, initial_features = init['best_model'], init['initial_features']

X, y, _ = select_xy(labeled, initial_features)
boruta = BorutaSelector(
    max_iter=settings.ml.boruta_max_iter,
    random_state=settings.ml.random_state,
    estimator_name=best_model,
).fit(X, y)
boruta.save('data/artifacts/boruta_features_best_model.json')
print(f"Q4 -- Boruta ({best_model}) kept {len(boruta.features_)} of {len(initial_features)} initial features:")
print(boruta.features_)

Xb, yb, panelb = select_xy(labeled, boruta.features_)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
boruta_summary, boruta_models = compare_models(
    Xb, yb, panelb, models=(best_model,), splitter=splitter, random_state=settings.ml.random_state
)
print("\nQ0/Q4 -- modeling results with Boruta-selected features:")
display(boruta_summary.round(4))
print("\nfold-by-fold detail:")
display(fold_metrics_table(boruta_models[best_model].fold_metrics_).round(4))

_ = Path('data/artifacts/model_comparison_boruta.json').write_text(
    json.dumps(
        {'n_features': len(boruta.features_), 'summary': boruta_summary.reset_index().to_dict(orient='records')},
        indent=2,
    ),
    encoding='utf-8',
)

## M14 — VIF

In [ ]:
!python -m ml_scan.cli ml vif --in data/artifacts/labeled_smoke.parquet --features data/artifacts/boruta_features.json --out data/artifacts/selected_features.json --max-vif 10

### M14b — Modeling results after VIF pruning (Q5 rationale + Q6)

**Why VIF, and why now**: Boruta already removes pure-noise columns; most of what survives is still overlapping transforms of the same handful of price/volume series (three Bollinger Band columns, `ATR` vs. `ATRr_14`, `RSI_14` vs. `MFI_14`, ...). Variance Inflation Factor pruning is the right second pass for exactly this failure mode: unlike a plain pairwise-correlation filter, VIF measures how well *each* remaining column is predicted by *all the others combined*, so it catches multi-column redundancy a correlation matrix alone would miss. `VIFPruner` (`ml_engine/selector.py`) also pre-drops any pair correlated above 0.95 before computing VIF, which keeps the regression well-conditioned. Fewer, less-redundant columns mean the tree models spend their limited splits on distinct signals instead of several near-copies of the same one.

The cell below re-fits the M12b winner on the final `selected_features.json` list (Boruta → VIF) and lines the results up against M12b (initial) and M13b (Boruta) so the effect of each stage is visible side by side.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from ml_scan.config import load_settings
from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.estimator import compare_models
from ml_scan.ml_engine.metrics import fold_metrics_table
from ml_scan.ml_engine.selector import load_feature_list
from ml_scan.ml_engine.splitter import PurgedWalkForward

settings = load_settings()
labeled = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
init = json.loads(Path('data/artifacts/model_comparison_initial.json').read_text())
best_model, initial_features = init['best_model'], init['initial_features']
boruta_info = json.loads(Path('data/artifacts/model_comparison_boruta.json').read_text())
selected = load_feature_list('data/artifacts/selected_features.json')

print(f"Q5 -- feature counts by stage: initial={len(initial_features)} -> boruta={boruta_info['n_features']} -> vif={len(selected)}")
print(selected)

Xs, ys, panels = select_xy(labeled, selected)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
vif_summary, vif_models = compare_models(
    Xs, ys, panels, models=(best_model,), splitter=splitter, random_state=settings.ml.random_state
)
print("\nQ0/Q6 -- modeling results with VIF-selected features:")
display(vif_summary.round(4))
print("\nfold-by-fold detail:")
display(fold_metrics_table(vif_models[best_model].fold_metrics_).round(4))

stages = pd.DataFrame(
    [
        {'stage': f"initial ({len(initial_features)} features)", **next(r for r in init['summary'] if r['model_key'] == best_model)},
        {'stage': f"boruta ({boruta_info['n_features']} features)", **boruta_info['summary'][0]},
        {'stage': f"vif ({len(selected)} features)", **vif_summary.reset_index().to_dict(orient='records')[0]},
    ]
).set_index('stage')[['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'roc_auc']]
print("\nQ6 -- selection-stage comparison (same model, shrinking feature set):")
display(stages.round(4))

_ = Path('data/artifacts/model_comparison_vif.json').write_text(
    json.dumps({'n_features': len(selected), 'summary': vif_summary.reset_index().to_dict(orient='records')}, indent=2),
    encoding='utf-8',
)

## M15 — Purged walk-forward

In [ ]:
!python -m pytest tests/test_purged_split.py -q

### M15b — Which stocks land in each train/test fold (Q1, completed)

The split used everywhere in this notebook is **time-based, not stock-based**: `PurgedWalkForward` slices the timeline into ordered blocks with an embargo gap between them, so every stock with rows inside a given window eventually appears on *both* sides — there is no fixed "training stocks" vs. "held-out stocks" split, only "earlier time" (train) vs. "later time, after the embargo" (test) per fold. The table below shows, fold by fold, exactly which smoke-universe stocks contributed train rows vs. test rows, and how many.

In [ ]:
import pandas as pd

from ml_scan.config import load_settings
from ml_scan.features.qc import select_xy, symbol_row_counts
from ml_scan.ml_engine.selector import load_feature_list
from ml_scan.ml_engine.splitter import PurgedWalkForward, fold_symbol_table

settings = load_settings()
labeled = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
selected = load_feature_list('data/artifacts/selected_features.json')
_, _, panel = select_xy(labeled, selected)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)

print("Q1 -- stocks fed into the model, with resolved-label row counts per stock:")
display(symbol_row_counts(labeled))

fold_table = fold_symbol_table(panel, splitter)
pivot = (
    fold_table.pivot_table(index='symbol', columns='fold', values=['n_train_rows', 'n_test_rows'], aggfunc='sum')
    .fillna(0)
    .astype(int)
)
print(f"\n{fold_table['symbol'].nunique()} stocks appear across {fold_table['fold'].nunique()} purged walk-forward folds; rows per stock per fold (train vs test):")
display(pivot)

## M16 — Train LightGBM

Display fold metrics table.

In [ ]:
!python -m ml_scan.cli ml train --in data/artifacts/labeled_smoke.parquet --features data/artifacts/selected_features.json --out data/artifacts/model.joblib
import json, pandas as pd
from pathlib import Path
meta = json.loads(Path('data/artifacts/model.json').read_text())
display(pd.DataFrame(meta.get('fold_metrics', [])))

### M16a — Hyperparameter optimization (Q7)

**Method: randomized search** (`sklearn.model_selection.RandomizedSearchCV`), scored on the *same* purged, embargoed walk-forward folds used everywhere in this notebook — the folds are passed in directly as `cv=[(train_idx, test_idx), ...]`, so the search cannot reward parameters that only look good on shuffled or leaky folds (sklearn's default K-fold would do exactly that on time-ordered data). The search space (`ml_engine/optimize.py::PARAM_DISTRIBUTIONS`) covers tree depth, learning rate, tree count, subsampling, and regularization for whichever model won M12b.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from ml_scan.config import load_settings
from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.optimize import random_search
from ml_scan.ml_engine.selector import load_feature_list
from ml_scan.ml_engine.splitter import PurgedWalkForward

settings = load_settings()
labeled = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
init = json.loads(Path('data/artifacts/model_comparison_initial.json').read_text())
best_model = init['best_model']
selected = load_feature_list('data/artifacts/selected_features.json')

Xs, ys, panels = select_xy(labeled, selected)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
best_params, search_results = random_search(
    Xs, ys, panels, model_name=best_model, splitter=splitter, n_iter=20, random_state=settings.ml.random_state
)
print(f"Q7 -- optimization method: RandomizedSearchCV on purged walk-forward folds, model={best_model!r}")
print("best_params:")
print(json.dumps(best_params, indent=2, default=str))

Path('data/artifacts/best_params.json').write_text(
    json.dumps({'model_name': best_model, 'best_params': best_params}, indent=2, default=str), encoding='utf-8'
)
print("\ntop parameter combinations tried, by mean ROC-AUC across folds:")
display(search_results.head(10))

### M16b — Retrain with the tuned hyperparameters (Q8, final model)

This is the model the rest of the notebook (M17 scan, M20/M21 backtest) actually uses from here on: the M12b winner retrained on the VIF-selected features with the hyperparameters found in M16a, saved over `data/artifacts/model.joblib`. Reported with the full metric set, fold by fold and as a fold-weighted average, so it is directly comparable to M12b/M13b/M14b above.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from ml_scan.config import load_settings
from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.artifacts import save_model
from ml_scan.ml_engine.estimator import MLEstimator
from ml_scan.ml_engine.metrics import fold_metrics_table
from ml_scan.ml_engine.selector import load_feature_list
from ml_scan.ml_engine.splitter import PurgedWalkForward

settings = load_settings()
labeled = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
init = json.loads(Path('data/artifacts/model_comparison_initial.json').read_text())
best_model = init['best_model']
best_params = json.loads(Path('data/artifacts/best_params.json').read_text())['best_params']
selected = load_feature_list('data/artifacts/selected_features.json')

Xs, ys, panels = select_xy(labeled, selected)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
tuned = MLEstimator(model=best_model, random_state=settings.ml.random_state, params=best_params)
tuned.fit_walk_forward(Xs, ys, panels, splitter)

print(f"Q8 -- final tuned {best_model} fold metrics:")
display(fold_metrics_table(tuned.fold_metrics_).round(4))
print("\nfold-weighted average:")
print(json.dumps(tuned.aggregate_metrics(), indent=2))

dest = save_model(tuned, 'data/artifacts/model.joblib')
print(f"\nwrote {dest} -- M17 (scan) and M20/M21 (backtest, Q9) below now run on this tuned model.")

## M17 — Inference scan

In [ ]:
import subprocess, sys
import pandas as pd

subprocess.check_call([
    sys.executable, "-m", "ml_scan.cli", "scan",
    "--asof", "latest",
    "--universe", str(TRAINING_UNIVERSE),
    "--out", "data/artifacts/scan_latest.csv",
])
scan = pd.read_csv('data/artifacts/scan_latest.csv')
print(f'scan universe: {TRAINING_UNIVERSE}')
display(scan)

## M18 — Indian costs

In [ ]:
!python -m pytest tests/test_costs.py -q

## M19 — Next-open fill lag

In [ ]:
!python -m pytest tests/test_fill_lag.py -q

## M20 — Event-driven backtest

In [ ]:
import subprocess, sys
from pathlib import Path
import pandas as pd
from ml_scan.config import load_settings
from ml_scan.execution.scanner import InferenceScanner, history_signals

settings = load_settings()
lab = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
scanner = InferenceScanner(settings)
scanner.load_artifacts('data/artifacts/model.joblib', 'data/artifacts/selected_features.json')
hist = history_signals(scanner.score_panel(lab), threshold=settings.ml.score_threshold)
Path('data/artifacts/scan_history.parquet').parent.mkdir(parents=True, exist_ok=True)
hist.to_parquet('data/artifacts/scan_history.parquet', index=False)
print('scan_history rows', len(hist))

subprocess.check_call([
    sys.executable, "-m", "ml_scan.cli", "backtest", "run",
    "--signals", "data/artifacts/scan_history.parquet",
    "--start", SMOKE_START, "--end", SMOKE_END,
    "--out", "data/artifacts/bt_smoke",
])

## M21 — Metrics

In [ ]:
!python -m ml_scan.cli backtest metrics --run data/artifacts/bt_smoke
import pandas as pd
from ml_scan.reporting.charts import equity_figure
eq = pd.read_parquet('data/artifacts/bt_smoke/equity.parquet')
fig = equity_figure(eq)
fig.show()

## M22 — Reporting

In [ ]:
!python -m ml_scan.cli report --run data/artifacts/bt_smoke --out data/artifacts/report_smoke.html
from ml_scan.ml_engine.artifacts import load_model
from ml_scan.reporting.charts import importance_figure
imps = load_model('data/artifacts/model.joblib').feature_importances()
if not imps.empty:
    importance_figure(imps).show()

## M23 — CLI surface

In [ ]:
!python -m ml_scan.cli --help

## M24 — End-to-end smoke (run everything)

In [ ]:
!python -m ml_scan.cli e2e smoke